In [1]:
import os
import pandas as pd
from datetime import datetime
import pytz
import unicodedata

# === Date & Paths ===
eastern = pytz.timezone("US/Eastern")
today = datetime.now(eastern).date()
today_str = today.isoformat()

paths = {
    "spreads": f"../data/novig-odds/novig_spreads_{today_str}.csv",
    "pitching_stats": f"../data/pitching_stats/pitching_stats_{today_str}.csv",
    "standings": f"../data/standings/standings_{today_str}.csv",
    "team_batting": f"../data/team_batting/team_batting_{today_str}.csv",
    "team_pitching": f"../data/team_pitching/team_pitching_{today_str}.csv"
}

# === Load Data ===
for label, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

spreads_df = pd.read_csv(paths["spreads"])
pitching_stats_df = pd.read_csv(paths["pitching_stats"])
standings_df = pd.read_csv(paths["standings"])
batting_df = pd.read_csv(paths["team_batting"])
teamp_df = pd.read_csv(paths["team_pitching"])

# === Normalize ===
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Preprocess Spreads ===
spreads_df['fav_score'] = None
spreads_df['dog_score'] = None
spreads_df['home_score'] = None
spreads_df['away_score'] = None

# === Normalize team names ===
pitching_stats_df['Home Team Clean'] = pitching_stats_df['Home Team'].apply(normalize_str)
pitching_stats_df['Away Team Clean'] = pitching_stats_df['Away Team'].apply(normalize_str)

# === Map team abbreviations to full names ===
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KC": "Kansas City Royals",
    "KAN": "Kansas City Royals", "AZ": "Arizona Diamondbacks", "ARI": "Arizona Diamondbacks",
    "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres", "MIN": "Minnesota Twins",
    "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# Derive home and away teams
spreads_df['home_team'] = spreads_df.apply(lambda row: row['fav_team'] if row['home_favorite'] == 1 else row['dog_team'], axis=1)
spreads_df['away_team'] = spreads_df.apply(lambda row: row['dog_team'] if row['home_favorite'] == 1 else row['fav_team'], axis=1)
spreads_df['home_team_full'] = spreads_df['home_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['away_team_full'] = spreads_df['away_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['home_team_clean'] = spreads_df['home_team_full'].apply(normalize_str)
spreads_df['away_team_clean'] = spreads_df['away_team_full'].apply(normalize_str)

# === Assign Pitchers and Stats ===
output_rows = []

for _, row in spreads_df.iterrows():
    home = row['home_team_clean']
    away = row['away_team_clean']

    ps_row = pitching_stats_df[
        (pitching_stats_df['Home Team Clean'] == home) &
        (pitching_stats_df['Away Team Clean'] == away)
    ]

    if ps_row.empty:
        ps_row = pitching_stats_df[
            (pitching_stats_df['Home Team Clean'] == away) &
            (pitching_stats_df['Away Team Clean'] == home)
        ]
        flipped = not ps_row.empty
    else:
        flipped = False

    if not ps_row.empty:
        ps_row = ps_row.iloc[0]
        if flipped:
            home_pitcher, away_pitcher = ps_row['Away Starter'], ps_row['Home Starter']
            home_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }
            away_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
        else:
            home_pitcher, away_pitcher = ps_row['Home Starter'], ps_row['Away Starter']
            home_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
            away_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }

        if row['home_favorite'] == 1:
            fav_pitcher, dog_pitcher = home_pitcher, away_pitcher
            fav_stats, dog_stats = home_stats, away_stats
        else:
            fav_pitcher, dog_pitcher = away_pitcher, home_pitcher
            fav_stats, dog_stats = away_stats, home_stats
    else:
        fav_pitcher = dog_pitcher = "Unknown"
        fav_stats = dog_stats = {k: None for k in ['era', 'whip', 'so', 'ip', 'avg']}

    row = row.copy()
    row['fav_pitcher'] = fav_pitcher
    row['dog_pitcher'] = dog_pitcher
    for k in fav_stats:
        row[f'fav_pitcher_{k}'] = fav_stats[k]
        row[f'dog_pitcher_{k}'] = dog_stats[k]

    # === New SO/IP Stat ===
    row['fav_pitcher_so_in'] = (
        fav_stats['so'] / fav_stats['ip'] if fav_stats['so'] is not None and fav_stats['ip'] else None
    )
    row['dog_pitcher_so_in'] = (
        dog_stats['so'] / dog_stats['ip'] if dog_stats['so'] is not None and dog_stats['ip'] else None
    )

    output_rows.append(row)

merged_df = pd.DataFrame(output_rows)

# === Normalize and Join Team Stats ===
for df in [standings_df, batting_df, teamp_df]:
    df['team_clean'] = df['Tm'].apply(normalize_str)

def get_team_stat(team_abbr, df, column):
    full = team_name_map.get(team_abbr.upper(), "")
    clean = normalize_str(full)
    row = df[df['team_clean'] == clean]
    return row[column].values[0] if not row.empty else None

# Standings
merged_df['fav_win_pct'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
merged_df['dog_win_pct'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))

# Batting/ERA
merged_df['fav_ba'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['dog_ba'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['fav_era_p'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))
merged_df['dog_era_p'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))

# Additional standings columns
standings_columns = ['W','L','Strk','R','RA','SOS','SRS','pythWL','Luck','last10','last20','last30']
for col in standings_columns:
    merged_df[f'fav_{col}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))
    merged_df[f'dog_{col}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))

# === Convert win/loss streaks to signed integers ===
def convert_streak(s):
    if isinstance(s, str) and len(s) > 1:
        direction = 1 if s[0].upper() == 'W' else -1 if s[0].upper() == 'L' else 0
        try:
            count = int(s[1:])
            return direction * count
        except ValueError:
            return None
    return None

merged_df['fav_Strk'] = merged_df['fav_Strk'].apply(convert_streak)
merged_df['dog_Strk'] = merged_df['dog_Strk'].apply(convert_streak)

# === Convert pythWL to win percentages ===
def wl_str_to_pct(wl_str):
    if isinstance(wl_str, str) and '-' in wl_str:
        try:
            wins, losses = map(int, wl_str.split('-'))
            total = wins + losses
            return wins / total if total > 0 else None
        except:
            return None
    return None

merged_df['fav_pythWL_pct'] = merged_df['fav_pythWL'].apply(wl_str_to_pct)
merged_df['dog_pythWL_pct'] = merged_df['dog_pythWL'].apply(wl_str_to_pct)

# Optional: drop the original 'W-L' strings
merged_df.drop(columns=['fav_pythWL', 'dog_pythWL'], inplace=True)

# === Convert last10/20/30 to win percentages ===
def wl_to_pct(wl_str):
    if isinstance(wl_str, str) and '-' in wl_str:
        try:
            wins, losses = map(int, wl_str.split('-'))
            total = wins + losses
            return wins / total if total > 0 else None
        except:
            return None
    return None

for span in ['last10', 'last20', 'last30']:
    merged_df[f'fav_{span}_pct'] = merged_df[f'fav_{span}'].apply(wl_to_pct)
    merged_df[f'dog_{span}_pct'] = merged_df[f'dog_{span}'].apply(wl_to_pct)

# Optional: drop original string columns
merged_df.drop(columns=['fav_last10', 'dog_last10', 'fav_last20', 'dog_last20', 'fav_last30', 'dog_last30'], inplace=True)

# Additional batting columns
batting_columns = ['R/G','PA','AB','R','H','2B','3B','HR','RBI','SB','BB','SO','BA','OBP','SLG','OPS','OPS+']
for col in batting_columns:
    col_renamed = f'{col}_b' if col in ['R','H','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))

# === Convert counting stats to rates per AB (percentages) ===
def safe_divide(numerator, denominator):
    try:
        num = float(numerator)
        denom = float(denominator)
        return num / denom if denom > 0 else None
    except (ValueError, TypeError):
        return None

conversion_pairs = [
    ('fav_2B', 'fav_AB', 'fav_2B_pct'),
    ('dog_2B', 'dog_AB', 'dog_2B_pct'),
    ('fav_3B', 'fav_AB', 'fav_3B_pct'),
    ('dog_3B', 'dog_AB', 'dog_3B_pct'),
    ('fav_HR', 'fav_AB', 'fav_HR_pct'),
    ('dog_HR', 'dog_AB', 'dog_HR_pct'),
    ('fav_RBI', 'fav_AB', 'fav_RBI_pct'),
    ('dog_RBI', 'dog_AB', 'dog_RBI_pct'),
]

for stat_col, ab_col, pct_col in conversion_pairs:
    merged_df[pct_col] = merged_df.apply(lambda row: safe_divide(row[stat_col], row[ab_col]), axis=1)


# Additional pitching columns
pitching_columns = ['RA/G','ERA','SV','H','R','ER','HR','BB','SO','ERA+','FIP','WHIP','H9']
for col in pitching_columns:
    col_renamed = f'{col}_p' if col in ['H','R','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))

# === Final Ordering ===
first_cols = [
    'game_time_est', 'fav_team', 'fav_score', 'dog_team', 'dog_score',
    'home_team', 'home_score', 'away_team', 'away_score',
    'fav_line', 'dog_line', 'fav_price', 'dog_price',
    'fav_price_american', 'dog_price_american', 'home_favorite', 'market',
    'fav_pitcher', 'dog_pitcher',
    'fav_pitcher_era', 'dog_pitcher_era',
    'fav_pitcher_whip', 'dog_pitcher_whip',
    'fav_pitcher_so', 'dog_pitcher_so',
    'fav_pitcher_ip', 'dog_pitcher_ip',
    'fav_pitcher_avg', 'dog_pitcher_avg',
    'fav_win_pct', 'dog_win_pct', 'fav_ba', 'dog_ba', 'fav_era_p', 'dog_era_p'
]
remaining_cols = [col for col in merged_df.columns if col not in first_cols and col not in ['market_timestamp', 'home_team_full', 'away_team_full', 'home_team_clean', 'away_team_clean']]
merged_df = merged_df[first_cols + remaining_cols]

# === Format computed percentage/rate columns ===
pct_cols = [
    'fav_pitcher_so_in', 'dog_pitcher_so_in',
    'fav_pythWL_pct', 'dog_pythWL_pct',
    'fav_last30_pct', 'dog_last30_pct',
    'fav_2B_pct', 'dog_2B_pct',
    'fav_3B_pct', 'dog_3B_pct',
    'fav_HR_pct', 'dog_HR_pct',
    'fav_RBI_pct', 'dog_RBI_pct'
]

for col in pct_cols:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].apply(lambda x: round(x, 3) if pd.notnull(x) else None)

# === Output ===
print("\u2705 Final enriched training set (WAR excluded):")
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
display(merged_df)

# === Save CSV ===
output_path = f"../training-data/training-set/training_set_{today_str}.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged_df.to_csv(output_path, index=False)
print(f"\u2705 Enriched training set saved to {output_path}")

✅ Final enriched training set (WAR excluded):


,game_time_est,fav_team,fav_score,dog_team,dog_score,home_team,home_score,away_team,away_score,fav_line,dog_line,fav_price,dog_price,fav_price_american,dog_price_american,home_favorite,market,fav_pitcher,dog_pitcher,fav_pitcher_era,dog_pitcher_era,fav_pitcher_whip,dog_pitcher_whip,fav_pitcher_so,dog_pitcher_so,fav_pitcher_ip,dog_pitcher_ip,fav_pitcher_avg,dog_pitcher_avg,fav_win_pct,dog_win_pct,fav_ba,dog_ba,fav_era_p,dog_era_p,fav_pitcher_so_in,dog_pitcher_so_in,fav_W,dog_W,fav_L,dog_L,fav_Strk,dog_Strk,fav_R,dog_R,fav_RA,dog_RA,fav_SOS,dog_SOS,fav_SRS,dog_SRS,fav_Luck,dog_Luck,fav_pythWL_pct,dog_pythWL_pct,fav_last10_pct,dog_last10_pct,fav_last20_pct,dog_last20_pct,fav_last30_pct,dog_last30_pct,fav_R/G,dog_R/G,fav_PA,dog_PA,fav_AB,dog_AB,fav_R_b,dog_R_b,fav_H_b,dog_H_b,fav_2B,dog_2B,fav_3B,dog_3B,fav_HR,dog_HR,fav_RBI,dog_RBI,fav_SB,dog_SB,fav_BB_b,dog_BB_b,fav_SO_b,dog_SO_b,fav_BA,dog_BA,fav_OBP,dog_OBP,fav_SLG,dog_SLG,fav_OPS,dog_OPS,fav_OPS+,dog_OPS+,fav_2B_pct,dog_2B_pct,fav_3B_pct,dog_3B_pct,fav_HR_pct,dog_HR_pct,fav_RBI_pct,dog_RBI_pct,fav_RA/G,dog_RA/G,fav_ERA,dog_ERA,fav_SV,dog_SV,fav_H_p,dog_H_p,fav_R_p,dog_R_p,fav_ER,dog_ER,fav_BB_p,dog_BB_p,fav_SO_p,dog_SO_p,fav_ERA+,dog_ERA+,fav_FIP,dog_FIP,fav_WHIP,dog_WHIP,fav_H9,dog_H9
0,2025-06-16T19:35:00-04:00,TB,None,BAL,None,BAL,None,TB,None,-1.5,1.5,0.345,0.683,190,-215,0,TB -1.5,Ryan Pepiot,Zach Eflin,3.31,4.08,1.14,1.09,73.0,36.0,81.2,53.0,0.230,0.252,0.549,0.429,.248,.241,3.45,4.89,0.899,0.679,39,30,32,40,3,3,4.6,4.0,3.7,5.0,0.1,-0.1,0.9,-1.1,-3.0,2.0,0.592,0.400,0.7,0.6,0.70,0.70,0.667,0.500,4.58,3.99,2650,2583,2380,2331,325,279,591,562,111,104,4,9,92,98,308,260,97,51,220,199,598,617,.248,.241,.316,.306,.393,.398,.708,.704,102,102,0.047,0.045,0.002,0.004,0.032,0.035,0.129,0.112,3.70,5.03,3.45,4.89,17,17,557,635,263,352,244,333,193,227,569,574,113,79,4.21,4.50,1.179,1.405,7.9,9.3
1,2025-06-16T19:05:00-04:00,NYY,None,LAA,None,LAA,None,NYY,None,-1.5,1.5,0.459,0.570,118,-133,0,NYY -1.5,Clarke Schmidt,José Soriano,3.60,3.86,1.24,1.50,57.0,64.0,55.0,79.1,0.221,0.269,0.600,0.471,.256,.226,3.61,4.82,1.036,0.809,42,33,28,37,-3,-3,5.3,4.2,3.8,5.1,0.0,-0.1,1.4,-1.0,-3.0,4.0,0.643,0.414,0.5,0.5,0.60,0.40,0.633,0.533,5.29,4.17,2714,2556,2371,2321,370,292,606,525,123,94,11,7,69,93,356,281,52,33,281,177,614,682,.256,.226,.339,.288,.455,.402,.794,.690,121,93,0.052,0.040,0.005,0.003,0.046,0.043,0.150,0.121,3.81,5.06,3.61,4.82,21,18,505,640,267,354,248,329,235,278,651,527,113,86,3.64,4.80,1.196,1.494,7.3,9.4
2,2025-06-16T21:40:00-04:00,SEA,None,BOS,None,SEA,None,BOS,None,-1.5,1.5,0.426,0.597,135,-148,1,SEA -1.5,Logan Gilbert,Lucas Giolito,2.37,5.45,0.79,1.54,44.0,31.0,30.1,39.2,0.164,0.301,0.514,0.507,.243,.253,3.96,3.96,1.462,0.791,36,37,34,36,3,5,4.4,4.8,4.4,4.5,-0.2,0.1,-0.2,0.4,1.0,-2.0,0.500,0.534,0.4,0.8,0.35,0.50,0.467,0.500,4.40,4.84,2707,2841,2391,2533,308,353,581,642,93,141,4,10,81,71,298,338,71,62,254,253,618,675,.243,.253,.323,.328,.398,.422,.722,.750,111,109,0.039,0.056,0.002,0.004,0.038,0.035,0.125,0.133,4.43,4.53,3.96,3.96,19,19,616,626,310,331,279,290,216,241,567,614,93,105,4.09,3.85,1.312,1.317,8.7,8.6
3,2025-06-16T18:45:00-04:00,WAS,None,COL,None,WAS,None,COL,None,-1.5,1.5,0.445,0.571,125,-133,1,WAS -1.5,Jake Irvin,Carson Palmquist,4.21,7.77,1.25,1.82,54.0,16.0,83.1,22.0,0.243,0.310,0.423,0.197,.239,.224,4.90,5.62,0.650,0.727,30,14,41,57,-8,1,4.2,3.4,5.0,6.2,0.0,0.3,-0.9,-2.5,1.0,-3.0,0.408,0.239,0.1,0.3,0.35,0.25,0.433,0.233,4.15,3.37,2615,2605,2355,2362,295,239,564,530,116,112,10,23,73,91,280,235,58,32,198,196,539,691,.239,.224,.307,.289,.381,.369,.688,.658,96,77,0.049,0.047,0.004,0.010,0.028,0.026,0.119,0.099,5.04,6.23,4.90,5.62,19,10,629,723,358,442,340,385,240,250,555,481,81,81,4.19,4.76,1.393,1.579,9.1,10.6
4,2025-06-16T18:40:00-04:00,PHI,None,MIA,None,MIA,None,PHI,None,-1.5,1.5,0.389,0.622,157,-165,0,MIA +1.5,Mick Abel,Sandy Alcantara,2.35,7.14,1.11,1.49,14.0,50.0,15.1,63.0,0.241,0.258,0.592,0.406,.257,.252,3.90,5.03,0.927,0.794,42,28,29,41,4,3,4.7,4.1,4.2,5.4

✅ Enriched training set saved to ../training-data/training-set/training_set_2025-06-16.csv


In [18]:
import os
import pandas as pd
from datetime import datetime
import pytz
import unicodedata

# === Date & Paths ===
# Override for June 11, 2025
today_str = "2025-06-02"

paths = {
    "spreads": "../data/novig-odds/novig_spreads_2025-06-02.csv",
    "pitching_stats": "../data/pitching_stats/pitching_stats_2025-06-02.csv",
    "standings": "../data/standings/standings_2025-06-02.csv",
    "team_batting": "../data/team_batting/team_batting_2025-06-02.csv",
    "team_pitching": "../data/team_pitching/team_pitching_2025-06-02.csv"
}

# === Load Data ===
for label, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

spreads_df = pd.read_csv(paths["spreads"])
pitching_stats_df = pd.read_csv(paths["pitching_stats"])
standings_df = pd.read_csv(paths["standings"])
batting_df = pd.read_csv(paths["team_batting"])
teamp_df = pd.read_csv(paths["team_pitching"])

# === Normalize ===
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Preprocess Spreads ===
spreads_df['fav_score'] = None
spreads_df['dog_score'] = None
spreads_df['home_score'] = None
spreads_df['away_score'] = None

# === Normalize team names ===
pitching_stats_df['Home Team Clean'] = pitching_stats_df['Home Team'].apply(normalize_str)
pitching_stats_df['Away Team Clean'] = pitching_stats_df['Away Team'].apply(normalize_str)

# === Map team abbreviations to full names ===
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KC": "Kansas City Royals",
    "KAN": "Kansas City Royals", "AZ": "Arizona Diamondbacks", "ARI": "Arizona Diamondbacks",
    "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres", "MIN": "Minnesota Twins",
    "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# Derive home and away teams
spreads_df['home_team'] = spreads_df.apply(lambda row: row['fav_team'] if row['home_favorite'] == 1 else row['dog_team'], axis=1)
spreads_df['away_team'] = spreads_df.apply(lambda row: row['dog_team'] if row['home_favorite'] == 1 else row['fav_team'], axis=1)
spreads_df['home_team_full'] = spreads_df['home_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['away_team_full'] = spreads_df['away_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['home_team_clean'] = spreads_df['home_team_full'].apply(normalize_str)
spreads_df['away_team_clean'] = spreads_df['away_team_full'].apply(normalize_str)

# === Assign Pitchers and Stats ===
output_rows = []

for _, row in spreads_df.iterrows():
    home = row['home_team_clean']
    away = row['away_team_clean']

    ps_row = pitching_stats_df[
        (pitching_stats_df['Home Team Clean'] == home) &
        (pitching_stats_df['Away Team Clean'] == away)
    ]

    if ps_row.empty:
        ps_row = pitching_stats_df[
            (pitching_stats_df['Home Team Clean'] == away) &
            (pitching_stats_df['Away Team Clean'] == home)
        ]
        flipped = not ps_row.empty
    else:
        flipped = False

    if not ps_row.empty:
        ps_row = ps_row.iloc[0]
        if flipped:
            home_pitcher, away_pitcher = ps_row['Away Starter'], ps_row['Home Starter']
            home_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }
            away_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
        else:
            home_pitcher, away_pitcher = ps_row['Home Starter'], ps_row['Away Starter']
            home_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
            away_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }

        if row['home_favorite'] == 1:
            fav_pitcher, dog_pitcher = home_pitcher, away_pitcher
            fav_stats, dog_stats = home_stats, away_stats
        else:
            fav_pitcher, dog_pitcher = away_pitcher, home_pitcher
            fav_stats, dog_stats = away_stats, home_stats
    else:
        fav_pitcher = dog_pitcher = "Unknown"
        fav_stats = dog_stats = {k: None for k in ['era', 'whip', 'so', 'ip', 'avg']}

    row = row.copy()
    row['fav_pitcher'] = fav_pitcher
    row['dog_pitcher'] = dog_pitcher
    for k in fav_stats:
        row[f'fav_pitcher_{k}'] = fav_stats[k]
        row[f'dog_pitcher_{k}'] = dog_stats[k]

    # === New SO/IP Stat ===
    row['fav_pitcher_so_in'] = (
        fav_stats['so'] / fav_stats['ip'] if fav_stats['so'] is not None and fav_stats['ip'] else None
    )
    row['dog_pitcher_so_in'] = (
        dog_stats['so'] / dog_stats['ip'] if dog_stats['so'] is not None and dog_stats['ip'] else None
    )

    output_rows.append(row)

merged_df = pd.DataFrame(output_rows)

# === Normalize and Join Team Stats ===
for df in [standings_df, batting_df, teamp_df]:
    df['team_clean'] = df['Tm'].apply(normalize_str)

def get_team_stat(team_abbr, df, column):
    full = team_name_map.get(team_abbr.upper(), "")
    clean = normalize_str(full)
    row = df[df['team_clean'] == clean]
    return row[column].values[0] if not row.empty else None

# Standings
merged_df['fav_win_pct'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
merged_df['dog_win_pct'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))

# Batting/ERA
merged_df['fav_ba'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['dog_ba'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['fav_era_p'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))
merged_df['dog_era_p'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))

# Additional standings columns
standings_columns = ['W','L','Strk','R','RA','SOS','SRS','pythWL','Luck','last10','last20','last30']
for col in standings_columns:
    merged_df[f'fav_{col}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))
    merged_df[f'dog_{col}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))

# === Convert win/loss streaks to signed integers ===
def convert_streak(s):
    if isinstance(s, str) and len(s) > 1:
        direction = 1 if s[0].upper() == 'W' else -1 if s[0].upper() == 'L' else 0
        try:
            count = int(s[1:])
            return direction * count
        except ValueError:
            return None
    return None

merged_df['fav_Strk'] = merged_df['fav_Strk'].apply(convert_streak)
merged_df['dog_Strk'] = merged_df['dog_Strk'].apply(convert_streak)

# === Convert pythWL to win percentages ===
def wl_str_to_pct(wl_str):
    if isinstance(wl_str, str) and '-' in wl_str:
        try:
            wins, losses = map(int, wl_str.split('-'))
            total = wins + losses
            return wins / total if total > 0 else None
        except:
            return None
    return None

merged_df['fav_pythWL_pct'] = merged_df['fav_pythWL'].apply(wl_str_to_pct)
merged_df['dog_pythWL_pct'] = merged_df['dog_pythWL'].apply(wl_str_to_pct)

# Optional: drop the original 'W-L' strings
merged_df.drop(columns=['fav_pythWL', 'dog_pythWL'], inplace=True)

# === Convert last10/20/30 to win percentages ===
def wl_to_pct(wl_str):
    if isinstance(wl_str, str) and '-' in wl_str:
        try:
            wins, losses = map(int, wl_str.split('-'))
            total = wins + losses
            return wins / total if total > 0 else None
        except:
            return None
    return None

for span in ['last10', 'last20', 'last30']:
    merged_df[f'fav_{span}_pct'] = merged_df[f'fav_{span}'].apply(wl_to_pct)
    merged_df[f'dog_{span}_pct'] = merged_df[f'dog_{span}'].apply(wl_to_pct)

# Optional: drop original string columns
merged_df.drop(columns=['fav_last10', 'dog_last10', 'fav_last20', 'dog_last20', 'fav_last30', 'dog_last30'], inplace=True)

# Additional batting columns
batting_columns = ['R/G','PA','AB','R','H','2B','3B','HR','RBI','SB','BB','SO','BA','OBP','SLG','OPS','OPS+']
for col in batting_columns:
    col_renamed = f'{col}_b' if col in ['R','H','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))

# === Convert counting stats to rates per AB (percentages) ===
def safe_divide(numerator, denominator):
    try:
        num = float(numerator)
        denom = float(denominator)
        return num / denom if denom > 0 else None
    except (ValueError, TypeError):
        return None

conversion_pairs = [
    ('fav_2B', 'fav_AB', 'fav_2B_pct'),
    ('dog_2B', 'dog_AB', 'dog_2B_pct'),
    ('fav_3B', 'fav_AB', 'fav_3B_pct'),
    ('dog_3B', 'dog_AB', 'dog_3B_pct'),
    ('fav_HR', 'fav_AB', 'fav_HR_pct'),
    ('dog_HR', 'dog_AB', 'dog_HR_pct'),
    ('fav_RBI', 'fav_AB', 'fav_RBI_pct'),
    ('dog_RBI', 'dog_AB', 'dog_RBI_pct'),
]

for stat_col, ab_col, pct_col in conversion_pairs:
    merged_df[pct_col] = merged_df.apply(lambda row: safe_divide(row[stat_col], row[ab_col]), axis=1)


# Additional pitching columns
pitching_columns = ['RA/G','ERA','SV','H','R','ER','HR','BB','SO','ERA+','FIP','WHIP','H9']
for col in pitching_columns:
    col_renamed = f'{col}_p' if col in ['H','R','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))

# === Final Ordering ===
first_cols = [
    'game_time_est', 'fav_team', 'fav_score', 'dog_team', 'dog_score',
    'home_team', 'home_score', 'away_team', 'away_score',
    'fav_line', 'dog_line', 'fav_price', 'dog_price',
    'fav_price_american', 'dog_price_american', 'home_favorite', 'market',
    'fav_pitcher', 'dog_pitcher',
    'fav_pitcher_era', 'dog_pitcher_era',
    'fav_pitcher_whip', 'dog_pitcher_whip',
    'fav_pitcher_so', 'dog_pitcher_so',
    'fav_pitcher_ip', 'dog_pitcher_ip',
    'fav_pitcher_avg', 'dog_pitcher_avg',
    'fav_win_pct', 'dog_win_pct', 'fav_ba', 'dog_ba', 'fav_era_p', 'dog_era_p'
]
remaining_cols = [col for col in merged_df.columns if col not in first_cols and col not in ['market_timestamp', 'home_team_full', 'away_team_full', 'home_team_clean', 'away_team_clean']]
merged_df = merged_df[first_cols + remaining_cols]

# === Format computed percentage/rate columns ===
pct_cols = [
    'fav_pitcher_so_in', 'dog_pitcher_so_in',
    'fav_pythWL_pct', 'dog_pythWL_pct',
    'fav_last30_pct', 'dog_last30_pct',
    'fav_2B_pct', 'dog_2B_pct',
    'fav_3B_pct', 'dog_3B_pct',
    'fav_HR_pct', 'dog_HR_pct',
    'fav_RBI_pct', 'dog_RBI_pct'
]

for col in pct_cols:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].apply(lambda x: round(x, 3) if pd.notnull(x) else None)

# === Output ===
print("\u2705 Final enriched training set (WAR excluded):")
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
display(merged_df)

# === Save CSV ===
output_path = f"../training-data/training-set/training_set_{today_str}.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged_df.to_csv(output_path, index=False)
print(f"\u2705 Enriched training set saved to {output_path}")

✅ Final enriched training set (WAR excluded):


,game_time_est,fav_team,fav_score,dog_team,dog_score,home_team,home_score,away_team,away_score,fav_line,dog_line,fav_price,dog_price,fav_price_american,dog_price_american,home_favorite,market,fav_pitcher,dog_pitcher,fav_pitcher_era,dog_pitcher_era,fav_pitcher_whip,dog_pitcher_whip,fav_pitcher_so,dog_pitcher_so,fav_pitcher_ip,dog_pitcher_ip,fav_pitcher_avg,dog_pitcher_avg,fav_win_pct,dog_win_pct,fav_ba,dog_ba,fav_era_p,dog_era_p,fav_pitcher_so_in,dog_pitcher_so_in,fav_W,dog_W,fav_L,dog_L,fav_Strk,dog_Strk,fav_R,dog_R,fav_RA,dog_RA,fav_SOS,dog_SOS,fav_SRS,dog_SRS,fav_Luck,dog_Luck,fav_pythWL_pct,dog_pythWL_pct,fav_last10_pct,dog_last10_pct,fav_last20_pct,dog_last20_pct,fav_last30_pct,dog_last30_pct,fav_R/G,dog_R/G,fav_PA,dog_PA,fav_AB,dog_AB,fav_R_b,dog_R_b,fav_H_b,dog_H_b,fav_2B,dog_2B,fav_3B,dog_3B,fav_HR,dog_HR,fav_RBI,dog_RBI,fav_SB,dog_SB,fav_BB_b,dog_BB_b,fav_SO_b,dog_SO_b,fav_BA,dog_BA,fav_OBP,dog_OBP,fav_SLG,dog_SLG,fav_OPS,dog_OPS,fav_OPS+,dog_OPS+,fav_2B_pct,dog_2B_pct,fav_3B_pct,dog_3B_pct,fav_HR_pct,dog_HR_pct,fav_RBI_pct,dog_RBI_pct,fav_RA/G,dog_RA/G,fav_ERA,dog_ERA,fav_SV,dog_SV,fav_H_p,dog_H_p,fav_R_p,dog_R_p,fav_ER,dog_ER,fav_BB_p,dog_BB_p,fav_SO_p,dog_SO_p,fav_ERA+,dog_ERA+,fav_FIP,dog_FIP,fav_WHIP,dog_WHIP,fav_H9,dog_H9
0,2025-06-02T21:45:00-04:00,SF,None,SD,None,SD,None,SF,None,-1.5,1.5,0.439,0.575,128,-135,0,SF -1.5,Logan Webb,Stephen Kolek,2.82,4.11,1.21,1.30,84.0,24.0,73.1,30.2,0.257,0.259,0.559,0.579,.233,.249,3.11,3.61,1.149,0.795,33,33,26,24,1,1,4.2,4.2,3.5,4.0,-0.1,-0.2,0.6,0.1,-1.0,3.0,0.576,0.526,0.4,0.6,0.45,0.45,0.467,0.533,4.20,4.25,2192,2110,1948,1887,248,242,454,470,86,84,9,7,41,60,237,222,34,42,204,178,501,392,.233,.249,.310,.316,.374,.385,.684,.701,95,96,0.044,0.045,0.005,0.004,0.029,0.028,0.122,0.118,3.49,3.96,3.11,3.61,17,21,444,430,206,226,179,203,178,181,512,503,125,112,3.27,3.83,1.201,1.208,7.7,7.6
1,2025-06-02T22:05:00-04:00,MIN,None,OAK,None,OAK,None,MIN,None,-1.5,1.5,0.493,0.525,103,-111,0,OAK +1.5,Joe Ryan,Luis Severino,2.57,3.89,0.83,1.23,72.0,54.0,63.0,71.2,0.191,0.240,0.534,0.383,.239,.256,3.32,5.71,1.143,0.758,31,23,27,37,-2,-6,4.0,4.2,3.5,6.1,-0.1,0.0,0.5,-1.8,-2.0,3.0,0.569,0.333,0.4,0.1,0.65,0.10,0.633,0.267,4.02,4.22,2153,2281,1940,2060,233,253,463,527,91,97,7,7,55,91,223,244,31,32,160,190,476,482,.239,.256,.307,.321,.379,.417,.687,.738,92,108,0.047,0.047,0.004,0.003,0.029,0.036,0.115,0.118,3.50,6.05,3.32,5.71,13,14,450,580,203,363,188,337,138,236,515,471,124,72,3.43,5.04,1.153,1.536,7.9,9.8
2,2025-06-02T22:10:00-04:00,LAD,None,NYM,None,NYM,None,LAD,None,-1.5,1.5,0.443,0.572,126,-134,0,LAD -1.5,Dustin May,Paul Blackburn,4.20,NaN,1.24,NaN,58.0,NaN,55.2,NaN,0.233,NaN,0.610,0.627,.268,.246,4.16,2.85,1.051,NaN,36,37,23,22,-1,3,5.8,4.4,4.5,3.3,-0.1,-0.1,1.3,1.0,-1.0,0.0,0.627,0.627,0.6,0.8,0.50,0.60,0.567,0.567,5.81,4.41,2288,2225,2019,1948,343,260,542,480,102,99,9,13,76,40,330,251,38,45,225,213,483,455,.268,.246,.345,.329,.472,.414,.817,.743,131,112,0.051,0.051,0.004,0.007,0.048,0.034,0.163,0.129,4.46,3.27,4.16,2.85,18,18,479,434,263,193,243,166,208,213,542,526,95,135,4.21,3.41,1.306,1.232,8.2,7.4


✅ Enriched training set saved to ../training-data/training-set/training_set_2025-06-02.csv


In [19]:
import os
import pandas as pd

# Directory with daily training set files
training_set_dir = "../training-data/training-set"

# Output path for combined file
combined_output_file = "../training-data/combined_training_set_from_june_2.csv"

# Earliest date to include
start_date = pd.to_datetime("2025-06-02")

# Collect matching DataFrames
all_dataframes = []

for file in sorted(os.listdir(training_set_dir)):
    if file.startswith("training_set_") and file.endswith(".csv"):
        # Extract date from filename
        date_str = file[len("training_set_"):-len(".csv")]
        try:
            file_date = pd.to_datetime(date_str)
            if file_date >= start_date:
                file_path = os.path.join(training_set_dir, file)
                print(f"📄 Including: {file}")
                df = pd.read_csv(file_path)
                df["source_file"] = file  # Optional for traceability
                all_dataframes.append(df)
            else:
                print(f"⏭️ Skipping (before June 2): {file}")
        except Exception as e:
            print(f"⚠️ Could not parse date from {file}: {e}")

# Combine and save
if all_dataframes:
    combined_df = pd.concat(all_dataframes, ignore_index=True)
    combined_df.to_csv(combined_output_file, index=False)
    print(f"\n✅ Combined training set saved to: {combined_output_file}")
    print(f"📊 Rows: {combined_df.shape[0]} | Columns: {combined_df.shape[1]}")
else:
    print("❌ No valid training set files found after June 2.")


⏭️ Skipping (before June 2): training_set_2025-05-29.csv
⏭️ Skipping (before June 2): training_set_2025-05-30.csv
⏭️ Skipping (before June 2): training_set_2025-05-31.csv
⏭️ Skipping (before June 2): training_set_2025-06-01.csv
📄 Including: training_set_2025-06-02.csv
📄 Including: training_set_2025-06-03.csv
📄 Including: training_set_2025-06-04.csv
📄 Including: training_set_2025-06-05.csv
📄 Including: training_set_2025-06-06.csv
📄 Including: training_set_2025-06-07.csv
📄 Including: training_set_2025-06-08.csv
📄 Including: training_set_2025-06-09.csv
📄 Including: training_set_2025-06-10.csv
📄 Including: training_set_2025-06-11.csv
📄 Including: training_set_2025-06-12.csv

✅ Combined training set saved to: ../training-data/combined_training_set_from_june_2.csv
📊 Rows: 112 | Columns: 128
